# Retrieved Papers Review by Client and Query Type

Use this notebook to inspect which papers were retrieved by each API client (`source`) and each query expansion strategy (`query_type`).

Data source: `../intermediate_outputs/step2_raw_papers.json`

In [7]:
from __future__ import annotations

import csv
import json
from collections import Counter, defaultdict
from pathlib import Path

In [8]:
DATA_PATH = Path('../intermediate_outputs/step5_fulltext_clean_papers.json')

with DATA_PATH.open('r', encoding='utf-8') as f:
    papers = json.load(f)

print(f'Loaded {len(papers)} raw paper records from {DATA_PATH}.')

Loaded 193 raw paper records from ..\intermediate_outputs\step5_fulltext_clean_papers.json.


In [9]:
def norm_text(value: object, fallback: str = 'unknown') -> str:
    if value is None:
        return fallback
    text = str(value).strip().lower()
    return text if text else fallback

grouped = defaultdict(list)
source_counts = Counter()
query_type_counts = Counter()
error_count = 0

for paper in papers:
    source = norm_text(paper.get('source'))
    query_type = norm_text(paper.get('query_type'))
    grouped[(source, query_type)].append(paper)
    source_counts[source] += 1
    query_type_counts[query_type] += 1
    if paper.get('is_error'):
        error_count += 1

print(f'Total records: {len(papers)}')
print(f'Error placeholder records: {error_count}')
print(f'Unique (source, query_type) groups: {len(grouped)}')

Total records: 193
Error placeholder records: 0
Unique (source, query_type) groups: 11


In [10]:
def print_table(headers: list[str], rows: list[list[object]]) -> None:
    widths = [len(str(h)) for h in headers]
    for row in rows:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(str(cell)))

    def fmt(row: list[object]) -> str:
        return ' | '.join(str(cell).ljust(widths[i]) for i, cell in enumerate(row))

    print(fmt(headers))
    print('-+-'.join('-' * w for w in widths))
    for row in rows:
        print(fmt(row))

sources = sorted(source_counts)
query_types = sorted(query_type_counts)

headers = ['source'] + query_types + ['total']
rows = []
for source in sources:
    row = [source]
    total_for_source = 0
    for qtype in query_types:
        count = len(grouped.get((source, qtype), []))
        row.append(count)
        total_for_source += count
    row.append(total_for_source)
    rows.append(row)

print('Counts by source and query_type')
print_table(headers, rows)

print('\nCounts by source')
for source, count in source_counts.most_common():
    print(f'- {source}: {count}')

print('\nCounts by query_type')
for qtype, count in query_type_counts.most_common():
    print(f'- {qtype}: {count}')

Counts by source and query_type
source           | concept_strings | structured_boolean | total
-----------------+-----------------+--------------------+------
crossref         | 7               | 34                 | 41   
elsevier         | 10              | 11                 | 21   
europe_pmc       | 7               | 32                 | 39   
openalex         | 7               | 19                 | 26   
scopus           | 10              | 21                 | 31   
semantic_scholar | 35              | 0                  | 35   

Counts by source
- crossref: 41
- europe_pmc: 39
- semantic_scholar: 35
- scopus: 31
- openalex: 26
- elsevier: 21

Counts by query_type
- structured_boolean: 117
- concept_strings: 76


In [11]:
for (source, qtype) in sorted(grouped):
    items = grouped[(source, qtype)]
    items_sorted = sorted(
        items,
        #sort by title
        key=lambda p: (p.get('title') or '').lower()
    )

    print('')
    print(f'=== {source} | {qtype} ({len(items)} papers) ===')
    for i, p in enumerate(items_sorted, start=1):
        year = p.get('year') if p.get('year') is not None else 'n/a'
        doi = p.get('doi') or 'no-doi'
        title = (p.get('title') or '').replace('\n', ' ').strip()
        print(f'{i:>2}. [{year}] {title} ({doi})')



=== crossref | concept_strings (7 papers) ===
 1. [1990] Expression Systems and Protein Production in Filamentous Fungi (10.1007/978-1-4613-1565-0_2)
 2. [2009] Heterologous Expression (10.1007/978-3-540-29678-2_2190)
 3. [n/a] Heterologous Expression in Yeast (10.1385/0-89603-321-x:341)
 4. [2009] Heterologous Expression of Membrane Proteins for Structural Analysis (10.1007/978-1-60761-344-2_1)
 5. [n/a] Heterologous Expression System (10.1007/3-540-29719-0_760)
 6. [1982] III. EVALUATION OF THE PROTEiN FOOD PRODUCTION 58 FOR CONSUMPTION PROJECT (10.1355/9789814376167-007)
 7. [2021] Structure-Based Identification of Potential Drugs Against FmtA of Staphylococcus aureus: Virtual Screening, Molecular Dynamics, MM-GBSA, and QM/MM (10.1007/s10930-020-09953-6)

=== crossref | structured_boolean (34 papers) ===
 1. [2021] An improved vector for baculovirus-mediated protein production in mammalian cells (10.1101/2021.10.18.464913)
 2. [n/a] Analysis of Heterologous Gene Expression in Xenop

In [12]:
# Optional: export a flat review file you can sort/filter elsewhere.
OUTPUT_CSV = Path('../intermediate_outputs/retrieved_papers_review.csv')
FIELDS = ['source', 'query_type', 'year', 'doi', 'title', 'url', 'is_error', 'response_time_ms']

with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=FIELDS)
    writer.writeheader()
    for paper in papers:
        writer.writerow({field: paper.get(field) for field in FIELDS})

print(f'Exported review CSV to: {OUTPUT_CSV.resolve()}')

Exported review CSV to: C:\Users\rebec\OneDrive\Documentos\TFM\repo\Pipetly\intermediate_outputs\retrieved_papers_review.csv
